In [0]:
def run_autoloader(
    volume_path: str,
    source_subpath: str,
    table_name: str,
    file_pattern: str = "*", 
    file_format: str = "json",
    catalog: str = "workspace",
    schema: str = "default",
    infer_schema: bool = True,
    trigger_available_now: bool = True,
    overwrite: bool = False
):
    """
    Ingestão Automática via Auto Loader (cloudFiles).
    Esta função identifica arquivos novos em um Volume e os carrega para uma tabela Delta.
    """
    from pyspark.sql import SparkSession
    spark = SparkSession.builder.getOrCreate()
    
    # --- CONFIGURAÇÃO DE CAMINHOS (Unity Catalog Volumes) ---
    # Define onde os dados brutos estão e onde o Spark salvará o progresso (metadados)
    base_path = f"/Volumes/{catalog}/{schema}/{volume_path}"
    source_path = f"{base_path}/{source_subpath}"
    
    # O checkpoint é o "cérebro" do processo: ele anota quais arquivos já foram processados
    checkpoint_path = f"{base_path}/checkpoints/{table_name}"
    # O schema_path armazena a estrutura da tabela detectada automaticamente
    schema_path = f"{base_path}/schemas/{table_name}"
    
    target_table = f"{catalog}.{schema}.{table_name}"

    # --- 🔄 LÓGICA DE REINICIALIZAÇÃO (OVERWRITE) ---
    # Se overwrite=True, deletamos o passado para reprocessar tudo do zero.
    # Para uma implantação de produção, seria interessante analisar a possibilidade de uma tabela append, não será inserido nesse projeto pois seria necessária uma etapa de validação de linhas duplicadas.
    if overwrite:
        print(f"Modo Overwrite ativado. Limpando metadados e tabela {target_table}...")
        
        # 1. Remove a tabela existente no catálogo
        spark.sql(f"DROP TABLE IF EXISTS {target_table}")
        
        # 2. IMPORTANTE: Apaga os diretórios de controle. 
        # Sem apagar o checkpoint, o Spark acharia que não há nada novo para ler.
        dbutils.fs.rm(checkpoint_path, True)
        dbutils.fs.rm(schema_path, True)
    
    # --- CONFIGURAÇÃO DO LEITOR (STREAMING) ---
    reader = (
        spark.readStream
        .format("cloudFiles")                     # Ativa o Auto Loader
        .option("cloudFiles.format", file_format) # Formato do arquivo (JSON, CSV, etc)
        .option("cloudFiles.schemaLocation", schema_path) # Onde guardar a evolução do schema
        .option("pathGlobFilter", file_pattern)   # Filtra arquivos específicos (ex: "*.json")
    )
    
    # Se ativado, o Spark tenta descobrir o tipo de dado de cada coluna sozinho
    if infer_schema:
        reader = reader.option("cloudFiles.inferColumnTypes", "true")
    
    # Carrega o fluxo de dados (lazy evaluation - ainda não processa nada)
    df = reader.load(source_path)
    
    # --- CONFIGURAÇÃO DO GRAVADOR (WRITER) ---
    writer = (
        df.writeStream
        .format("delta")                          # Salva no formato otimizado Delta
        .option("checkpointLocation", checkpoint_path) # Essencial para não ler arquivos duplicados
        .outputMode("append")                     # Adiciona novos dados à tabela existente
    )
    
    # Trigger 'availableNow' faz o Spark processar todo o lote atual e parar (estilo Batch)
    if trigger_available_now:
        writer = writer.trigger(availableNow=True)
    
    # Inicia a gravação na tabela final
    query = writer.toTable(target_table)
    
    # Se for um processo agendado, aguardamos o fim do processamento antes de encerrar o job
    if trigger_available_now:
        query.awaitTermination()
    
    return f"Tabela {target_table} {'sobrescrita' if overwrite else 'atualizada'} com sucesso."